In [2]:
# Food Delivery Data Analysis

# This notebook loads CSV, JSON, and SQL data, merges them using keys,
# and performs analysis for the hackathon.


In [3]:
import pandas as pd
import sqlite3


In [4]:
orders = pd.read_csv("orders.csv")
orders.head()


,order_id,user_id,restaurant_id,order_date,total_amount,restaurant_name
0,1,2508,450,18-02-2023,842.97,New Foods Chinese
1,2,2693,309,18-01-2023,546.68,Ruchi Curry House Multicuisine
2,3,2084,107,15-07-2023,163.93,Spice Kitchen Punjabi
3,4,319,224,04-10-2023,1155.97,Darbar Kitchen Non-Veg
4,5,1064,293,25-12-2023,1321.91,Royal Eatery South Indian


In [5]:
users = pd.read_json("users.json")
users.head()


,user_id,name,city,membership
0,1,User_1,Chennai,Regular
1,2,User_2,Pune,Gold
2,3,User_3,Bangalore,Gold
3,4,User_4,Bangalore,Regular
4,5,User_5,Pune,Gold


In [6]:
conn = sqlite3.connect(":memory:")

with open("restaurants.sql", "r") as file:
    sql_script = file.read()

conn.executescript(sql_script)

restaurants = pd.read_sql("SELECT * FROM restaurants", conn)
restaurants.head()


,restaurant_id,restaurant_name,cuisine,rating
0,1,Restaurant_1,Chinese,4.8
1,2,Restaurant_2,Indian,4.1
2,3,Restaurant_3,Mexican,4.3
3,4,Restaurant_4,Chinese,4.1
4,5,Restaurant_5,Chinese,4.8


In [7]:
orders_users = pd.merge(
    orders,
    users,
    on="user_id",
    how="left"
)

orders_users.head()


,order_id,user_id,restaurant_id,order_date,total_amount,restaurant_name,name,city,membership
0,1,2508,450,18-02-2023,842.97,New Foods Chinese,User_2508,Hyderabad,Regular
1,2,2693,309,18-01-2023,546.68,Ruchi Curry House Multicuisine,User_2693,Pune,Regular
2,3,2084,107,15-07-2023,163.93,Spice Kitchen Punjabi,User_2084,Chennai,Gold
3,4,319,224,04-10-2023,1155.97,Darbar Kitchen Non-Veg,User_319,Bangalore,Gold
4,5,1064,293,25-12-2023,1321.91,Royal Eatery South Indian,User_1064,Pune,Regular


In [8]:
final_df = pd.merge(
    orders_users,
    restaurants,
    on="restaurant_id",
    how="left"
)

final_df.head()


,order_id,user_id,restaurant_id,order_date,total_amount,restaurant_name_x,name,city,membership,restaurant_name_y,cuisine,rating
0,1,2508,450,18-02-2023,842.97,New Foods Chinese,User_2508,Hyderabad,Regular,Restaurant_450,Mexican,3.2
1,2,2693,309,18-01-2023,546.68,Ruchi Curry House Multicuisine,User_2693,Pune,Regular,Restaurant_309,Indian,4.5
2,3,2084,107,15-07-2023,163.93,Spice Kitchen Punjabi,User_2084,Chennai,Gold,Restaurant_107,Mexican,4.0
3,4,319,224,04-10-2023,1155.97,Darbar Kitchen Non-Veg,User_319,Bangalore,Gold,Restaurant_224,Chinese,4.8
4,5,1064,293,25-12-2023,1321.91,Royal Eatery South Indian,User_1064,Pune,Regular,Restaurant_293,Italian,3.0


In [9]:
final_df.shape


(10000, 12)

In [10]:
final_df.to_csv("final_food_delivery_dataset.csv", index=False)


In [11]:
final_df.shape[0]


10000

In [12]:
final_df["membership"].value_counts()


membership
Regular    5013
Gold       4987
Name: count, dtype: int64

In [13]:
final_df.groupby("city")["total_amount"].sum().sort_values(ascending=False)


city
Bangalore    2206946.58
Chennai      1990513.03
Pune         1924797.93
Hyderabad    1889366.58
Name: total_amount, dtype: float64

In [14]:
final_df[final_df["membership"] == "Gold"]["total_amount"].mean()


np.float64(797.1455564467616)

In [15]:
## Conclusion

# - Data from CSV, JSON, and SQL formats was successfully loaded
# - Left joins were used to retain all order records
# - Gold members contribute significantly to revenue
# - Metro cities like Bangalore generate higher revenue


In [16]:
gold_df = final_df[final_df["membership"] == "Gold"]


In [17]:
gold_revenue_by_city = (
    gold_df
    .groupby("city")["total_amount"]
    .sum()
    .sort_values(ascending=False)
)

gold_revenue_by_city


city
Chennai      1080909.79
Pune         1003012.32
Bangalore     994702.59
Hyderabad     896740.19
Name: total_amount, dtype: float64

In [18]:
avg_order_value_by_cuisine = (
    final_df
    .groupby("cuisine")["total_amount"]
    .mean()
    .sort_values(ascending=False)
)

avg_order_value_by_cuisine


cuisine
Mexican    808.021344
Italian    799.448578
Indian     798.466011
Chinese    798.389020
Name: total_amount, dtype: float64

In [19]:
user_total_spend = (
    final_df
    .groupby("user_id")["total_amount"]
    .sum()
)


In [20]:
users_above_1000 = user_total_spend[user_total_spend > 1000]
users_above_1000.count()


np.int64(2544)

In [21]:

bins = [3.0, 3.5, 4.0, 4.5, 5.0]
labels = ["3.0–3.5", "3.6–4.0", "4.1–4.5", "4.6–5.0"]

final_df["rating_range"] = pd.cut(
    final_df["rating"],
    bins=bins,
    labels=labels,
    include_lowest=True
)


In [22]:
revenue_by_rating_range = (
    final_df
    .groupby("rating_range")["total_amount"]
    .sum()
    .sort_values(ascending=False)
)

revenue_by_rating_range


C:\Users\Admin\AppData\Local\Temp\ipykernel_16164\3695441876.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("rating_range")["total_amount"]


rating_range
4.6–5.0    2197030.75
3.0–3.5    2136772.70
4.1–4.5    1960326.26
3.6–4.0    1717494.41
Name: total_amount, dtype: float64

In [23]:
gold_df = final_df[final_df["membership"] == "Gold"]


In [24]:
gold_avg_order_value_by_city = (
    gold_df
    .groupby("city")["total_amount"]
    .mean()
    .sort_values(ascending=False)
)

gold_avg_order_value_by_city


city
Chennai      808.459080
Hyderabad    806.421034
Bangalore    793.223756
Pune         781.162243
Name: total_amount, dtype: float64

In [25]:
gold_avg_order_value_by_city.idxmax()


'Chennai'

In [26]:
restaurant_count_by_cuisine = (
    final_df
    .groupby("cuisine")["restaurant_id"]
    .nunique()
    .sort_values()
)

restaurant_count_by_cuisine


cuisine
Chinese    120
Indian     126
Italian    126
Mexican    128
Name: restaurant_id, dtype: int64

In [28]:
total_orders = final_df.shape[0]
total_orders


10000

In [29]:
gold_orders = final_df[final_df["membership"] == "Gold"].shape[0]
gold_orders


4987

In [30]:
gold_percentage = (gold_orders / total_orders) * 100
round(gold_percentage)


50

In [37]:
final_df.columns



Index(['order_id', 'user_id', 'restaurant_id', 'order_date', 'total_amount',
       'restaurant_name_x', 'name', 'city', 'membership', 'restaurant_name_y',
       'cuisine', 'rating', 'rating_range'],
      dtype='object')

In [38]:
restaurant_stats = (
    final_df
    .groupby("restaurant_name_x")
    .agg(
        total_orders=("order_id", "count"),
        avg_order_value=("total_amount", "mean")
    )
)


In [39]:
restaurant_stats[
    restaurant_stats["total_orders"] < 20
].sort_values(
    by="avg_order_value",
    ascending=False
).head(10)


,total_orders,avg_order_value
restaurant_name_x,,
Hotel Dhaba Multicuisine,13,1040.222308
Sri Mess Punjabi,12,1029.180833
Ruchi Biryani Punjabi,16,1002.140625
Sri Delights Pure Veg,18,989.467222
Classic Kitchen Family Restaurant,19,973.167895
Hotel Dhaba Chinese,18,973.125556
Amma Mess Pure Veg,18,965.299444
Hotel Biryani Pure Veg,13,964.577692
Annapurna Curry House Multicuisine,17,954.512353


In [40]:
combo_revenue = (
    final_df
    .groupby(["membership", "cuisine"])["total_amount"]
    .sum()
    .sort_values(ascending=False)
)

combo_revenue


membership  cuisine
Regular     Mexican    1072943.30
            Italian    1018424.75
Gold        Mexican    1012559.79
            Italian    1005779.05
Regular     Indian      992100.27
Gold        Indian      979312.31
            Chinese     977713.74
Regular     Chinese     952790.91
Name: total_amount, dtype: float64

In [55]:
final_df["order_date"] = pd.to_datetime(final_df["order_date"])


In [56]:
final_df["quarter"] = final_df["order_date"].dt.to_period("Q")



In [57]:
quarter_revenue = (
    final_df
    .groupby("quarter")["total_amount"]
    .sum()
    .sort_values(ascending=False)
)

quarter_revenue


quarter
2023Q3    2037385.10
2023Q4    2018263.66
2023Q1    1993425.14
2023Q2    1945348.72
2024Q1      17201.50
Freq: Q-DEC, Name: total_amount, dtype: float64

In [58]:
quarter_revenue.idxmax()


Period('2023Q3', 'Q-DEC')

In [46]:
gold_orders_count = final_df[final_df["membership"] == "Gold"].shape[0]
gold_orders_count



4987

In [47]:
hyderabad_df = final_df[final_df["city"] == "Hyderabad"]
hyderabad_revenue = hyderabad_df["total_amount"].sum()
round(hyderabad_revenue)



1889367

In [48]:
distinct_users = final_df["user_id"].nunique()
distinct_users


2883

In [49]:
gold_df = final_df[final_df["membership"] == "Gold"]
gold_avg_order_value = gold_df["total_amount"].mean()
round(gold_avg_order_value, 2)


np.float64(797.15)

In [50]:
high_rating_df = final_df[final_df["rating"] >= 4.5]
high_rating_orders_count = high_rating_df.shape[0]
high_rating_orders_count


3374

In [51]:
gold_df = final_df[final_df["membership"] == "Gold"]
gold_revenue_by_city = (
    gold_df
    .groupby("city")["total_amount"]
    .sum()
    .sort_values(ascending=False)
)

gold_revenue_by_city



city
Chennai      1080909.79
Pune         1003012.32
Bangalore     994702.59
Hyderabad     896740.19
Name: total_amount, dtype: float64

In [52]:
top_city = gold_revenue_by_city.idxmax()
top_city
orders_in_top_city = gold_df[gold_df["city"] == top_city].shape[0]
orders_in_top_city


1337